# Laboratorio 5 - Análisis de paquetes y detección de anomalías
## Security Data Science
### Edwin Andrés Ortega Kou - 22305

### Scapy 

In [ ]:
from scapy.all import * 
import pandas as pd 
import numpy as np
import binascii 
import seaborn as sns
sns.set(color_codes=True)
%matplotlib inline

### Sniffing de tráfico

In [ ]:

num_of_packets_to_sniff = 10
pcap = sniff(count=num_of_packets_to_sniff)

# rdpcap returns packet list
## packetlist object can be enumerated
print(type(pcap))
print(len(pcap))
print(pcap)
pcap[0]

## Leer un PCAP de Wireshark

In [ ]:
pcap = rdpcap("analisis_paquetes.pcap")

## Otras funciones

In [ ]:
# high-level functions are already coded
lsc()

### Recordemos las capas: Física/Enlace -> Red/IP -> Transporte

In [ ]:
ethernet_frame = pcap[0]
ip_packet = ethernet_frame.payload
segment = ip_packet.payload
data = segment.payload # Retrieve payload that comes after layer 4

In [ ]:
# Understanding the object types in scapy
print(type(ethernet_frame))
print(type(ip_packet))
print(type(segment))

In [ ]:
hexdump(pcap[0])

In [ ]:
ls(pcap[0])

In [ ]:
pcap[0].summary()

## Convertir PCAP a un DataFrame

Scapy es una librería muy versatil, pero sus estructuras de datos no son tan fáciles de manipular como un data frame

In [ ]:
# Collect field names from IP/TCP/UDP (These will be columns in DF)
ip_fields = [field.name for field in IP().fields_desc]
tcp_fields = [field.name for field in TCP().fields_desc]
udp_fields = [field.name for field in UDP().fields_desc]

dataframe_fields = ip_fields + ['time'] + tcp_fields + ['payload','payload_raw','payload_hex']

# Create blank DataFrame
df = pd.DataFrame(columns=dataframe_fields)
for packet in pcap[IP]:
    # Field array for each row of DataFrame
    field_values = []
    # Add all IP fields to dataframe
    for field in ip_fields:
        if field == 'options':
            # Retrieving number of options defined in IP Header
            field_values.append(len(packet[IP].fields[field]))
        else:
            field_values.append(packet[IP].fields[field])
    
    field_values.append(packet.time)
    
    layer_type = type(packet[IP].payload)
    for field in tcp_fields:
        try:
            if field == 'options':
                field_values.append(len(packet[layer_type].fields[field]))
            else:
                field_values.append(packet[layer_type].fields[field])
        except:
            field_values.append(None)
    
    # Append payload
    field_values.append(len(packet[layer_type].payload))
    field_values.append(packet[layer_type].payload.original)
    field_values.append(binascii.hexlify(packet[layer_type].payload.original))
    # Add row to DF
    df_append = pd.DataFrame([field_values], columns=dataframe_fields)
    df = pd.concat([df, df_append], axis=0)

# Reset Index
df = df.reset_index()
# Drop old index column
df = df.drop(columns="index")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df

In [ ]:
pcap[IP]

In [ ]:
print(len(pcap))


## Estadísticas

In [ ]:
# Top Source Adddress
# Para datos categóricos, describe devuelve conteo, valores distintos, valor más frecuente (top) y su frecuencia
print("# Top Source Address")
print(df['src'].describe(),'\n\n') 

frequent_address = df['src'].describe()['top']

# Who is the top address speaking to
print("# Who is Top Address Speaking to?")
print(df[df['src'] == frequent_address]['dst'].unique(),"\n\n")



In [ ]:
# Unique Source Addresses
print("Unique Source Addresses")
print(df['src'].unique())

print()

# Unique Destination Addresses
print("Unique Destination Addresses")
print(df['dst'].unique())

In [ ]:
# Group by Source Address and Payload Sum
source_addresses = df.groupby("src")['payload'].sum()
source_addresses.plot(kind='barh',title="Addresses Sending Payloads",figsize=(8,5))

In [ ]:
features = df.columns.tolist()
features

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats


important_features = ['len', 'payload']


# ---- Panel 1: Histograms with Gaussian fit overlay ----
fig, axes = plt.subplots(1, 2, figsize=(24, 4))

for ax, feat in zip(axes, important_features):
    data = df[feat].values
    ax.hist(data, bins=35, density=True, alpha=0.6, color='#4A90D9', edgecolor='white')

    # Overlay the best-fit Gaussian curve
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'N({mu:.0f}, {sigma:.0f}²)')

    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Histogram vs. Gaussian Fit',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()